In [1]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt

def phase_to_m(phase, wv):
    """ Converts phase in radians to meters. 
    Parameters
    ----------
    phase : float or array of floats
        The phase input to be converted in radians. 
    wv: float
        The wavelength to use for conversion in meters. 
    Returns
    -------
    The phase information in units of meters.
    """
    return phase * wv / (2*np.pi)

def p_to_delta(P, f, D): 
    """ Converts P (the peak to valley error in the pupil plane) 
    induced by a delta (the longitudinal distance) offset.
    I.e., given some defocused image we can recontruct what that 
    distance would have been. Note that f and D always need to be
    in the same units -- mm is common, and P and delta will have 
    the same units. 
    Parameters
    ----------
    P : float
        The peak to valley error. 
    f : float
        The focal length leading up to this plane. 
    D : float
        The pupil size/telescope diameter of this plane. 
    Returns
    -------
    The delta defocus that would have been needed to create the P2V 
    we see. 
    """
    return 8 * P * (f/D)**2 


def delta_to_p(delta, f, D):
    """ p_to_delta() but in reverse. Note that f and D always need 
    to be in the same units -- mm is common, and P and delta will 
    have the same units. 
    Parameters
    ----------
    delta : float
        The longitudinal defocus distance. 
    f : float
        The focal length leading up to this plane. 
    D : float
        The pupil size of this plane. 
    Returns
    -------
    The P2V we would see based on the input delta. 
    """
    return -1*delta / (8 * (f/D)**2)

In [5]:
## Setting up
pupil_size = 10.12e-3 # KiloDM pupil = 10.12 mm
focal_length = 500e-3 # focal length into detector 500 mm
pupil_grid = make_pupil_grid(256, pupil_size)
aperture = circular_aperture(pupil_size)
telescope_pupil = aperture(pupil_grid)
wavefront = Wavefront(telescope_pupil, wavelength=650e-9)
focal_grid = make_focal_grid(q=16, num_airy=16, pupil_diameter=pupil_size, focal_length=focal_length, reference_wavelength=650e-9)
prop_p2f = FraunhoferPropagator(pupil_grid, focal_grid, focal_length=focal_length)
prop_f2p = FraunhoferPropagator(focal_grid, pupil_grid, focal_length=focal_length)


/var/folders/fp/0vqz18r15679y3w674kfj3k40000gq/T/ipykernel_23417/3153092937.py:4: DeprecationWarning: circular_aperture is deprecated. Its new name is make_circular_aperture.
  aperture = circular_aperture(pupil_size)


In [6]:
##Parameters
test_ab = 0.5*influence_functions[6] #Test abberation
example_defocus = influence_functions[4].shaped # Example Defocus
scales = [-10, -5, 5, 10]  # List of scales

NameError: name 'influence_functions' is not defined

In [ ]:
# Function to calculate defocus
def calculate_defocus(example_defocus, scale, f, D):
    defocus_phase = example_defocus * scale
    p2v_radians = np.max(defocus_phase) - np.min(defocus_phase)
    p2v_m = phase_to_m(p2v_radians, wavelength)
    delta = p_to_delta(p2v_m, f, D)
    return p2v_radians, delta


In [ ]:
# Function to generate PSF list
def generate_psf_list(scales, example_defocus, test_ab, telescope_pupil, wavelength):
    psf_list = []
    distance_list = []

    # Calculate and add the no-defocus image
    no_defocus_phase = np.zeros_like(example_defocus)
    no_defocus_image = propagate_image(no_defocus_phase, test_ab, telescope_pupil, wavelength)
    psf_list.append(no_defocus_image)
    
    # Calculate and add the defocus images
    for scale in scales:
        defocus_phase = example_defocus * scale
        p2v_radians = np.max(defocus_phase) - np.min(defocus_phase)
        p2v_m = phase_to_m(p2v_radians, wavelength)
        delta = p_to_delta(p2v_m, focal_length, pupil_size)
        defocus_image = propagate_image(defocus_phase, test_ab, telescope_pupil, wavelength)
        psf_list.append(defocus_image)
        distance_list.append(delta * 1e6)  # Convert to microns
    
    return psf_list, distance_list


In [ ]:
# Function to propagate and visualize the focal image
def propagate_image(defocus_phase, test_ab, telescope_pupil, wavelength):
    pupil_image = Wavefront(telescope_pupil, wavelength)  # Placeholder for the pupil image
    pupil_image.electric_field = np.exp(1j * (test_ab + defocus_phase.ravel())) * telescope_pupil
    focal_image = prop_p2f(pupil_image)
    imshow_field(np.log10(focal_image.intensity / focal_image.intensity.max()), vmin=-5)
    return np.array(focal_image.intensity)


# Example usage

# Size of image in microns
dx_list = [2.0071, 2.0071, 2.0071]

# Output the lists
print("PSF List:", psf_list)
print("Distance List:", distance_list)
print("DX List:", dx_list)